# 02 — Synthetic connection examples for both models and both NOUL designs

**Purpose:** understand binary and hybrid requests for JEV-9B and JEV-27B v1.1 before interpreting saved results. Section 9 displays all four model/design combinations with one shared synthetic state. Their responses are fabricated mock data, not screening performance.

The original one-state/one-question example and separate one-state/seven-question hybrid test remain below. Both paid flags are false; the binary walkthrough now selects a dedicated mock-only profile by default. Original live configuration files, API safeguards and saved responses are preserved. No API key is needed for the defaults; mock calls write no files.

Section 10 lists existing saved connection metadata without resending anything. Model responses already collected for evaluation are reused in notebook 03, then compared in 04–05. Hosted and future local backends remain separate; no weights, GPU packages or training are introduced.

## 1. Load the safe binary walkthrough configuration

Select the project's `.venv` kernel and restart after source changes. `BINARY_CONNECTION_CONFIG` below points to `comparison_binary.yaml`, which uses mock mode, one state, one question and no writes. This does not alter the original live profile `configs/connections/inference_sh.yaml`.

The comparison mock grid later selects both model YAMLs explicitly. For a deliberately approved original live test only, select the original live profile and review its model/YAML permissions plus the paid flag in Section 5. Never switch a four-example demonstration loop into live mode.

In [ ]:
try:
    from jev_eval.environment import initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select the project's .venv Python kernel. Run uv sync --locked --group notebooks "
        "from the repository terminal if dependencies are missing."
    ) from None

from jev_eval.connections import connection_readiness, load_connection

project = initialize_notebook()
BINARY_CONNECTION_CONFIG = "configs/connections/comparison_binary.yaml"
connection = load_connection(project.root, BINARY_CONNECTION_CONFIG)

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "model": connection.model.name,
            "description": connection.model.display_name,
            "app_id": connection.model.hosted.app_id,
            "connection_mode": connection.settings.execution.mode,
        }
    ]
)

## 2. Inspect prerequisites without submitting a request

The table shows the selected app and mode. Mock mode requires no credential. A deliberate live call uses `INFERENCE_API_KEY` from the environment, not `JEV_API_KEY`; its value must never be printed.

The app IDs are `autotrust/jev-9b` and `autotrust/jev-27b-v1-1`, selected through the corresponding model YAML. Changing a model deployment is distinct from changing binary versus hybrid question design.

In [ ]:
connection_readiness(connection)

## 3. Define one synthetic study and one connectivity-only question

This invented study is only for checking request/response handling. It is not part of the evaluation cohort. The simple `screening_eligibility` question below is not substituted for the archived eligibility instructions.

The later four-arm examples use their configured binary/hybrid templates with a shared synthetic file-based state. These examples differ in question content from this deliberately simple single-question test; neither is evidence of scientific screening accuracy.

In [ ]:
from jev_eval.connections import with_synthetic_example

state = {
    "title": "Effects of an antidepressant on depressive-like behaviour in mice",
    "abstract": (
        "Adult mice were randomly assigned to receive an "
        "antidepressant or vehicle control for 14 days. "
        "Depressive-like behaviour was measured using "
        "the forced swim test."
    ),
}
question = {
    "id": "screening_eligibility",
    "instructions": (
        "Does this study report an in vivo experiment "
        "in mice or rats evaluating an antidepressant "
        "intervention with a behavioural outcome?"
    ),
}


example_connection = with_synthetic_example(connection, state=state, question=question)

In [ ]:
pd.DataFrame(
    [
        {"question": name, "type": value.type, "instructions": value.instructions}
        for name, value in example_connection.task.questions.items()
    ]
)

## 4. Preview one request without sending it

The request contains the selected `app`, the synthetic `state`, `nouls: [question]`, and `wait: true`. Authentication is sent separately by the backend, so no credential appears in this preview. This is one state, one question, and one model, with no retries or batching.

In [ ]:
from jev_eval.connections import preview_connection

preview_connection(example_connection)

## 5. Run one example — mock by default; live use can be billable

With the default mock profile, the next cell returns a fabricated response without contacting the provider or writing files. `ALLOW_PAID_CALLS` remains false.

**Paid API warning:** deliberately selecting the original live profile and enabling this flag can send one request to `https://api.inference.sh/apps/run`. Each rerun can incur another charge. Live mode requires YAML paid/write permissions and automatically saves raw response bytes, validated answers and provenance. One state, one question, one POST and no retries remain the original limits. The API key is sent by the reusable backend, never embedded in the notebook request.

The cell shows HTTP status and credential-redacted JSON. Authentication/HTTP/schema failures have informative messages. A timeout can leave the provider outcome unknown; do not automatically resend. **If saving fails after a live response, keep the kernel open and use Section 6 for save-only recovery.**

In [ ]:
import json

from jev_eval.config import PrerequisiteError
from jev_eval.connections import (
    ConnectionPersistenceError,
    connection_response_view,
    run_connection,
    summarize_connection,
)

ALLOW_PAID_CALLS = False
result = None
if example_connection.settings.execution.mode == "mock" or ALLOW_PAID_CALLS:
    try:
        result = run_connection(example_connection, allow_paid_calls=ALLOW_PAID_CALLS)
    except ConnectionPersistenceError as error:
        result = error.result
        print("SAVE FAILED: the API result is retained in memory.")
        print("Do not rerun this request cell. Fix storage and use Section 6 to save only.")
    except PrerequisiteError as error:
        print(f"Connection not started: {error}")
    if result is not None:
        view = connection_response_view(result)
        print("HTTP status:", view["http_status"])
        print(view["message"])
        if view["is_json"]:
            print(json.dumps(view["raw_json"], indent=2, ensure_ascii=False))
        display(summarize_connection(result))
        if result.directory is not None:
            print("Saved artifacts:", result.directory)
else:
    print("Request skipped: ALLOW_PAID_CALLS is disabled.")

## 6. Recover an unsaved result without resending

Successful live calls are saved automatically; this step is only needed if Section 5 reports a storage failure. The retained `result` contains the original request, response, and provenance. Fix the local storage problem first and keep this kernel open.

**File-writing warning:** set `RECOVER_UNSAVED_RESULT = True` to write the retained artifacts. This cell makes no API request and creates no new model charge. It leaves an already saved result alone. If a failed write left a partial directory, use the new directory printed by the successful save as the complete record.

In [ ]:
from jev_eval.connections import save_connection

RECOVER_UNSAVED_RESULT = False
if result is None:
    print("No result is available in this kernel. Do not resend to recover an earlier response.")
elif result.directory is not None:
    print("Already saved:", result.directory)
elif not RECOVER_UNSAVED_RESULT:
    print("Save-only recovery is disabled. Enable it after fixing storage if needed.")
else:
    try:
        saved = save_connection(result, example_connection, allow_writes=True)
        print("Saved artifacts without another API request:", saved)
    except (OSError, PrerequisiteError):
        print(
            "Saving still failed; result remains in memory. Check storage and YAML write permission."
        )

## 7. From connectivity to the saved four-arm comparison

Connection checks are separate from real citation screening. A successful mock response demonstrates the software path only; a live synthetic response tests connectivity, not performance. Notebook 03 reuses existing binary and hybrid full archives for both models without resubmitting them. Notebook 04 evaluates saved probabilities; notebook 05 reports them together.

Local inference remains separate and unimplemented. No GPU packages or model weights are installed. Preserve executed copies privately, then clear outputs and execution counts before committing source notebooks.

## 8. Preview and test one synthetic hybrid request

The existing single-question connection test above retains its one-state/one-question safeguards. This separate profile explicitly permits **one synthetic state, seven NOULs and one POST**. `configs/connections/hybrid_v1.yaml` starts in mock mode with paid calls and writing off. Model selection remains in `model_config_file`; use either existing model YAML.

All seven fabricated mock probabilities are 0.8. They check connectivity/schema handling and are not model performance. In live mode, executing the next cell is potentially billable and automatically saves raw and normalized responses with provenance. Enable paid/write permissions in the hybrid connection YAML and `ALLOW_HYBRID_PAID_CALLS` deliberately; do not enable the old binary test to run hybrid. No retries or bulk loop are added. A successful response must contain all seven requested question IDs.

In [ ]:
from jev_eval.connections import load_hybrid_connection

hybrid_connection = load_hybrid_connection(project.root)
display(connection_readiness(hybrid_connection))
hybrid_request_preview = preview_connection(hybrid_connection)
print("Requested app:", hybrid_request_preview["app"])
print("Question IDs:", [q["id"] for q in hybrid_request_preview["input"]["nouls"]])

In [ ]:
ALLOW_HYBRID_PAID_CALLS = False
hybrid_connection_result = None
if hybrid_connection.settings.execution.mode == "mock" or ALLOW_HYBRID_PAID_CALLS:
    try:
        hybrid_connection_result = run_connection(
            hybrid_connection, allow_paid_calls=ALLOW_HYBRID_PAID_CALLS
        )
    except ConnectionPersistenceError as error:
        hybrid_connection_result = error.result
        print("Saving failed; keep this result and save it without resending the request.")
    except PrerequisiteError as error:
        print("Hybrid connection blocked:", error)
    if hybrid_connection_result is not None:
        display(summarize_connection(hybrid_connection_result))
        hybrid_view = connection_response_view(hybrid_connection_result)
        print("HTTP status:", hybrid_view["http_status"])
        print(hybrid_view["message"])
        if hybrid_view["is_json"]:
            print(json.dumps(hybrid_view["raw_json"], indent=2))
else:
    print("Live hybrid request skipped. No paid-call flag was enabled.")

## 9. Four explicit mock examples: binary/hybrid × JEV-9B/JEV-27B

`configs/connections/comparison.yaml` names both model YAMLs and the binary/hybrid mock profiles. All four use one shared synthetic state. Binary sends one NOUL; hybrid sends seven NOULs in a single POST. Both approaches use the same reusable API client and validation, with different configured question-count limits.

**Mock-only safeguard:** the loader rejects live mode or paid/write permissions in these profiles. The examples below cannot be converted into a four-request paid loop by changing a profile. Use the independent single-request sections above for any deliberate live check. Fabricated scores (0.8) and task/model names are demonstrators, not observed model performance. No API key or files are needed.

The binary configured template reflects the archived binary wording, while hybrid uses reviewed-v1 wording. The four-arm research comparison in notebooks 03–05 reports that difference rather than treating them as an identical eligibility definition.

In [ ]:
from jev_eval.connections import load_comparison_connections

mock_connections = load_comparison_connections(project.root)
connection_grid = pd.DataFrame(
    [
        {
            "approach": approach,
            "model": model,
            "requested_app": item.model.hosted.app_id,
            "mode": item.settings.execution.mode,
            "states": 1,
            "questions": len(item.task.questions),
            "max_POST_attempts": item.backend.max_attempts,
            "writes": item.settings.output.allow_writes,
        }
        for (approach, model), item in mock_connections.items()
    ]
)
display(connection_grid)

### 9.1 Binary NOUL — jev-9b

Preview the request, then inspect a **fabricated mock** JSON response. This code sends no request to Inference.sh and writes no files. No performance metric should be calculated from it.

In [ ]:
selected_mock = mock_connections[("binary", "jev-9b")]
print("MOCK ONLY — fabricated response, not JEV performance")
print(json.dumps(preview_connection(selected_mock), indent=2))
binary_jev_9b_mock_result = run_connection(selected_mock)
view = connection_response_view(binary_jev_9b_mock_result)
print("Mock HTTP status:", view["http_status"])
print(json.dumps(view["raw_json"], indent=2))
display(summarize_connection(binary_jev_9b_mock_result))

### 9.2 Binary NOUL — jev-27b

Preview the request, then inspect a **fabricated mock** JSON response. This code sends no request to Inference.sh and writes no files. No performance metric should be calculated from it.

In [ ]:
selected_mock = mock_connections[("binary", "jev-27b")]
print("MOCK ONLY — fabricated response, not JEV performance")
print(json.dumps(preview_connection(selected_mock), indent=2))
binary_jev_27b_mock_result = run_connection(selected_mock)
view = connection_response_view(binary_jev_27b_mock_result)
print("Mock HTTP status:", view["http_status"])
print(json.dumps(view["raw_json"], indent=2))
display(summarize_connection(binary_jev_27b_mock_result))

### 9.3 Hybrid NOUL — jev-9b

Preview the request, then inspect a **fabricated mock** JSON response. This code sends no request to Inference.sh and writes no files. No performance metric should be calculated from it.

In [ ]:
selected_mock = mock_connections[("hybrid", "jev-9b")]
print("MOCK ONLY — fabricated response, not JEV performance")
print(json.dumps(preview_connection(selected_mock), indent=2))
hybrid_jev_9b_mock_result = run_connection(selected_mock)
view = connection_response_view(hybrid_jev_9b_mock_result)
print("Mock HTTP status:", view["http_status"])
print(json.dumps(view["raw_json"], indent=2))
display(summarize_connection(hybrid_jev_9b_mock_result))

### 9.4 Hybrid NOUL — jev-27b

Preview the request, then inspect a **fabricated mock** JSON response. This code sends no request to Inference.sh and writes no files. No performance metric should be calculated from it.

In [ ]:
selected_mock = mock_connections[("hybrid", "jev-27b")]
print("MOCK ONLY — fabricated response, not JEV performance")
print(json.dumps(preview_connection(selected_mock), indent=2))
hybrid_jev_27b_mock_result = run_connection(selected_mock)
view = connection_response_view(hybrid_jev_27b_mock_result)
print("Mock HTTP status:", view["http_status"])
print(json.dumps(view["raw_json"], indent=2))
display(summarize_connection(hybrid_jev_27b_mock_result))

## 10. Inspect existing saved connections without repeating requests

The inventory reads only an allowlist of fields from existing connection metadata files and checks whether normalized/raw files are present. It neither constructs a live client nor resends any saved task. A metadata catalogue is not full validation or recovery. Older nonstandard recovery folders remain unchanged even if they lack a standard metadata file.

Saved connection checks are not evaluation records. Historical result bodies and their provenance remain under ignored `results/connections/`; notebooks 03–05 read the separate full-evaluation archives. Do not repeat a paid call simply to recreate an output display.

In [ ]:
from jev_eval.connections import saved_connection_inventory

saved_connections = saved_connection_inventory(project.root)
if saved_connections.empty:
    print(
        "No standard saved connection metadata files here; no request will be made to replace them."
    )
else:
    display(saved_connections)
print("Next: notebook 03 reviews/reuses all four full runs; notebooks 04–05 compare them.")